# Lesson 03 - Agentiske Designmønstre

I denne leksjonen utforsker vi tre grunnleggende designmønstre for å bygge effektive AI-agenter:

1. **Klare Agentinstruksjoner** — Utforme presise, rolledefinerende prompts som styrer agentens atferd  
2. **Strukturert Output med Pydantic-modeller** — Sikre at agenter returnerer forutsigbare, validerte data  
3. **Agenter med Ett Ansvarsområde** — Designe fokuserte agenter som hver gjør én ting godt  

Vi vil anvende hvert mønster på en **reise destinasjonsanbefaler**-scenario, og gradvis bygge et system som kan foreslå destinasjoner, sjekke tilgjengelighet og håndtere logistikk.


## Oppsett


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mønster 1: Klare instruksjoner til agenten

Det mest effektfulle mønsteret er også det enkleste: å skrive klare, detaljerte instruksjoner til agenten din.

Gode instruksjoner definerer:
- **Hvem** agenten er (personlighet og tone)
- **Hva** den skal gjøre (ansvarsområder steg for steg)
- **Hvordan** den skal oppføre seg (begrensninger og stil)

Nedenfor lager vi en reise-konsiergeagent med eksplisitte instruksjoner som former hvert svar den produserer.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Mønster 2: Strukturert utdata med Pydantic-modeller

Fritekst er nyttig for samtaler, men systemer videre i kjeden trenger strukturert data.  
Ved å kombinere **Pydantic-modeller** med en **verktøyfunksjon**, kan vi:

- Definere et nøyaktig skjema for agentens utdata  
- Validere svar automatisk  
- Integrere agentresultater pålitelig i applikasjonslogikken  

Vi introduserer også et verktøy som returnerer destinasjonsdetaljer, slik at agenten baserer anbefalingene sine på ekte data.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Mønster 3: Enkeltansvarsagenter

Komplekse oppgaver drar nytte av å dele arbeidet mellom flere fokuserte agenter, hver med et enkelt ansvar:

- En **Destinasjonsekspert** som kjenner til steder og tilgjengelighet
- En **Logistikkplanlegger** som håndterer fly, hoteller og reiseplaner

Dette speiler programvareingeniørprinsippet om *separasjon av bekymringer* — hver agent er lettere å teste, vedlikeholde og forbedre uavhengig.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Sammendrag

I denne leksjonen anvendte vi tre agentdesignmønstre på et reiseanbefalingsscenario:

| Mønster | Nøkkelidé | Fordel |
|---|---|---|
| **Klare instruksjoner** | Definer persona, ansvar og begrensninger på forhånd | Konsistent agentadferd i tråd med merkevaren |
| **Strukturert utdata** | Bruk Pydantic-modeller som responsformat | Validerte, maskinlesbare resultater |
| **Enkeltansvar** | Gi hver agent ett fokusert oppgave | Enklere å teste, vedlikeholde og komponere |

Disse mønstrene komponeres naturlig — du kan kombinere klare instruksjoner med strukturert utdata inne i en enkeltansvarsagent for å bygge robuste, produksjonsklare systemer.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
